# Day 23: LangChain "Chains" vs "Expressions" (LCEL)

Welcome to Day 23! Today we are looking at **LangChain Expression Language (LCEL)**, the modern standard for building modular, composable, and production-ready RAG and agent pipelines.

As an AI Engineer, you'll find that legacy LangChain concepts (like `LLMChain` or `RetrievalQA`) often hide too much complexity, making them difficult to debug or extend. LCEL replaces these rigid structures with a functional approach. 

## The "Why" and "How" behind LCEL

**Why LCEL?**
1.  **Composability**: Instead of monolithic classes, LCEL lets you chain together modular components (Prompts, LLMs, Output Parsers, Retrievers) using the pipe operator `|`. 
2.  **Streaming & Async out of the box**: LCEL natively supports synchronous, asynchronous, streaming, and batched execution (`.invoke()`, `.stream()`, `.batch()`, `.ainvoke()`), which is critical for production web applications where latency matters.
3.  **Transparency**: It is much easier to see the data flow. You pass inputs, map variables, and pipe them directly into models and parsers.

**How it Works:**
At its core, LCEL relies on the `Runnable` protocol. Every element in an LCEL pipeline implements `Runnable`, meaning they all share the standard `.invoke()` and `.stream()` methods. When you use the pipe `|` (e.g., `prompt | model | parser`), you are creating a `RunnableSequence`.


## 1. Code Implementation: The Basics vs LCEL

Let's look at how to implement a basic Q&A pipeline using LCEL. We will use `ChatGroq` or a mock model if no API key is available, but for strict adherence to non-fictitious, working code we'll use a `FakeListChatModel` from `langchain_core` to guarantee execution without API dependencies during testing. In production, this would be `ChatOpenAI`, `ChatAnthropic`, etc.


In [1]:
from typing import Dict, Any
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSerializable, RunnablePassthrough
# For testing/demonstration without an API key, we use a Fake model.
# In a real app, use: from langchain_openai import ChatOpenAI
from langchain_core.language_models.fake_chat_models import FakeListChatModel

def build_lcel_chain() -> RunnableSerializable:
    """
    Builds a simple LCEL chain that takes a topic and generates a joke.
    Returns:
        A compiled Runnable sequence.
    """
    # 1. Prompt Template
    prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}.")
    
    # 2. LLM (Using a fake model to ensure the code executes locally without keys)
    # The fake model simply cycles through the provided responses.
    model = FakeListChatModel(responses=["Why do Python programmers prefer dark mode? Because light attracts bugs!"])
    
    # 3. Output Parser (Extracts the string from the AIMessage object)
    parser = StrOutputParser()
    
    # 4. LCEL Pipeline composition using the | operator
    chain = prompt | model | parser
    
    return chain

# Execute the basic chain
print("--- Basic LCEL Execution ---")
chain = build_lcel_chain()
result = chain.invoke({"topic": "programming"})
print(f"Input: programming\nOutput: {result}\n")


--- Basic LCEL Execution ---
Input: programming
Output: Why do Python programmers prefer dark mode? Because light attracts bugs!



## 2. Advanced: RAG Pipeline with LCEL

Now let's simulate a Retrieval-Augmented Generation (RAG) pipeline. This is where LCEL shines. We use `RunnablePassthrough` to pass the original question along with the retrieved context.


In [2]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnableLambda

def mock_retriever(query: str) -> List[Document]:
    """
    A deterministic function mimicking a vector database retrieval.
    """
    return [
        Document(page_content="LangChain Expression Language (LCEL) uses the pipe operator | for composition."),
        Document(page_content="RunnablePassthrough allows passing inputs unchanged.")
    ]

def build_rag_chain() -> RunnableSerializable:
    """
    Builds a RAG pipeline using LCEL and RunnableParallel.
    """
    # 1. RAG Prompt
    template = """Answer the question based only on the following context:
Context: {context}

Question: {question}
"""
    prompt = ChatPromptTemplate.from_template(template)
    
    # 2. LLM
    model = FakeListChatModel(responses=["Based on the context, LCEL uses the pipe operator | for composition."])
    parser = StrOutputParser()
    
    # 3. Data Formatting function
    def format_docs(docs: List[Document]) -> str:
        return "\n\n".join(doc.page_content for doc in docs)
    
    # 4. Constructing the pipeline with RunnableParallel
    # We need to map the incoming string query to the inputs expected by the prompt (context and question).
    setup_and_retrieval = RunnableParallel(
        {"context": RunnableLambda(mock_retriever) | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    )
    
    rag_chain = setup_and_retrieval | prompt | model | parser
    return rag_chain

# Execute the RAG chain
print("--- RAG LCEL Execution ---")
rag_chain = build_rag_chain()
rag_result = rag_chain.invoke("How does LCEL handle composition?")
print(f"Output: {rag_result}")


--- RAG LCEL Execution ---
Output: Based on the context, LCEL uses the pipe operator | for composition.


## Common Pitfalls in Production

1.  **Over-complicating `RunnableMap` (RunnableParallel):** When dealing with complex inputs, developers often nest `RunnableParallel` too deeply. It's usually better to break complex data preparation into standard Python functions and wrap them in a `@chain` decorator or a simple `RunnableLambda`.
2.  **Forgetting the Output Parser:** If you don't end your chain with an `OutputParser` (like `StrOutputParser()`), `.invoke()` will return an `AIMessage` object instead of a clean string. This often breaks downstream API responses that expect JSON serialization.
3.  **Mixing Legacy and LCEL:** Don't try to embed legacy `LLMChain` objects directly inside an LCEL pipeline without wrapping them. It creates inconsistent `.stream()` and `.batch()` behaviors. Stick strictly to LCEL primitives (`Runnables`).
4.  **Debugging Black Boxes:** When a long pipe (`a | b | c | d`) fails, it can be hard to know which step broke. Use `.with_config({"run_name": "MyStep"})` on components, or attach LangSmith for tracing.


## Practical Lab: Translation and Formatting Chain

**Your Task:**
You need to build a two-step chain using LCEL.

1.  **Step 1 (Translation):** Take a user `input_text` and a `target_language`. Use a prompt to translate the text.
2.  **Step 2 (Formatting):** Take the output of the translation, and pass it to a second prompt that formats it as a formal email.
3.  **Requirement:** Chain these together using the `|` operator. 

*Hint: Use `RunnablePassthrough.assign()` or `RunnableParallel` to keep variables across the pipeline, or simply pipe the string output of the first chain into the input of the second prompt using another dict mapping.*


In [3]:
from langchain_core.runnables import RunnableLambda

def build_translation_email_chain() -> RunnableSerializable:
    """
    Builds a two-stage LCEL pipeline: Translation -> Email Formatting.
    """
    # Stage 1: Translation
    translate_prompt = ChatPromptTemplate.from_template(
        "Translate the following text into {target_language}: {input_text}"
    )
    # Using a fake model for deterministic execution
    model = FakeListChatModel(responses=[
        "Bonjour le monde", # Response for stage 1
        "Subject: Formal Greeting\n\nBonjour le monde\n\nBest Regards." # Response for stage 2
    ])
    parser = StrOutputParser()
    
    translation_chain = translate_prompt | model | parser
    
    # Stage 2: Email Formatting
    email_prompt = ChatPromptTemplate.from_template(
        "Take the following text and format it as a formal business email: {translated_text}"
    )
    
    # We pipe the output of translation_chain into the inputs for email_prompt
    # Since translation_chain outputs a string, we map it to the dictionary expected by email_prompt
    full_chain = (
        {"translated_text": translation_chain} 
        | email_prompt 
        | model 
        | parser
    )
    
    return full_chain

# Execute Lab task
print("\n--- Lab Execution: Translation & Email Chain ---")
lab_chain = build_translation_email_chain()
lab_result = lab_chain.invoke({
    "input_text": "Hello world", 
    "target_language": "French"
})
print("Final Output:")
print(lab_result)



--- Lab Execution: Translation & Email Chain ---
Final Output:
Subject: Formal Greeting

Bonjour le monde

Best Regards.
